# El cierre del flujo

Este notebook enseña el hito 5 de EDsim con el modelo ya calibrado (`config/`, tarea 5.6): qué hace
el abandono, de qué dependen los destinos y cuánto bloquean los ingresados. Son seis análisis:
el abandono frente a la saturación, los destinos frente a la gravedad real G, los boxes
bloqueados con la ley de Little, la semana de enero antes y después, el contrafactual de quien se
fue y el estimador de Kaplan–Meier de la paciencia.

Toda la lógica vive en el paquete `edsim` y está testeada (`edsim.runs`); aquí solo se llama a
sus funciones, se dibuja y se explica.

## 1. Cómo usar este notebook

- **Kernel:** en Cursor, pulsa *Select Kernel* y elige el entorno de `.venv` del proyecto.
  Si no aparece, ejecuta antes `uv sync --all-packages` en la terminal.
- **Ejecutar:** usa *Restart & Run All*. El notebook funciona de principio a fin sin
  depender del orden en que ejecutaste las celdas a mano.
- **Semillas:** la ejecución usa `SEMILLA = 2026`. Con la misma semilla el resultado es idéntico;
  cámbiala para ver otra muestra del azar.
- **Duración:** en modo normal tarda unos 45 segundos; casi todo son las dos ejecuciones de
  91 días (la del hito 4 y la del hito 5). Para ir más rápido, baja `DIAS` en la celda de
  parámetros.
- **Modo rápido:** con la variable de entorno `EDSIM_NB_RAPIDO=1` las ejecuciones duran 9 días y
  «la semana» pasa a ser la de los días 1 a 7, con el mismo código. Tarda menos de 10 segundos, es lo
  que hace el CI (`EDSIM_NB_RAPIDO=1 uv run pytest --nbmake notebooks/`) y los números de ese
  modo no sirven para sacar conclusiones: algunos niveles se quedan sin pacientes y las curvas
  salen ruidosas. **Las cifras que cita el texto son las de la ejecución normal** (91 días,
  semilla 2026).
- **Las gráficas desaparecen tras cada commit:** el hook `nbstripout` de pre-commit borra
  resultados e imágenes antes de guardar el notebook en git. Vuelve a ejecutarlo para verlas.
- **El modelo** es el del hito 5 de `config/`: los perfiles con el bloque de abandono y destinos
  (paciencia con k = 4), el centro con 36 boxes y la espera de cama de mediana 210 min, y la
  plantilla de la 4.6c. Para la comparación de la sección 6 se lee también el modelo del hito 4,
  que desde la tarea 5.6 vive congelado en `tests/data/` (`centro_h4.toml` y `perfiles_h4.toml`,
  los que fueron `config/`). El [notebook 04](04_perfiles_clinicos.ipynb) enseña ese modelo y
  usa las mismas copias. Se ejecuta desde la carpeta `notebooks/`, de ahí las rutas
  `../config/` y `../tests/data/`.
- **Memoria:** cada ejecución se reduce a su tabla de episodios, sus curvas y su ground truth
  nada más simularla, y sus eventos se liberan.

In [ ]:
import os
import resource
import time
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from edsim.arrivals.profile import load_profile
from edsim.arrivals.rate import MINUTES_PER_DAY
from edsim.arrivals.report import WEEKDAYS
from edsim.calibration.evaluate import DRAIN_DAYS
from edsim.engine.analysis import episode_table, step_curves
from edsim.engine.center import load_center
from edsim.engine.engine import simulate_ed
from edsim.engine.events import MemorySink
from edsim.engine.report import MUTED_COLOR, OBSERVED_COLOR, SECOND_COLOR, THEORY_COLOR
from edsim.profiles.config import DESTINATIONS, REASON_KEYS, load_profiles
from edsim.profiles.outcome import (
    destination_probabilities,
    outcome_model,
    patience_quantile,
)
from edsim.runs import (
    abandonment_by_arrival_hour,
    abandonment_counterfactual,
    bed_little,
    bed_occupancy,
    clock_mean,
    curve_steps,
    destinations_by_severity,
    kaplan_meier,
    left_without_being_seen_shares,
    patience_survival,
    true_level_abandonment,
)

INICIO_NB = time.perf_counter()

# --- Parámetros ------------------------------------------------------------------------
# Con EDSIM_NB_RAPIDO=1 (el CI) todo se reduce para que el notebook tarde poco.
RAPIDO = os.environ.get("EDSIM_NB_RAPIDO") == "1"

CONFIG = Path("../config")  # el notebook se ejecuta desde notebooks/
H4 = Path("../tests/data")  # el config/ del hito 4, congelado en la tarea 5.6
LLEGADAS = load_profile(CONFIG / "llegadas.toml")  # el lunes 5 de enero de 2026 es el día 0
CENTRO_H5 = load_center(CONFIG / "centro.toml")
PERFILES_H5 = load_profiles(CONFIG / "perfiles.toml")
CENTRO_H4 = load_center(H4 / "centro_h4.toml")
PERFILES_H4 = load_profiles(H4 / "perfiles_h4.toml")

SEMILLA = 2026
CAL = 1 if RAPIDO else 2  # días de calentamiento
DIAS = 9 if RAPIDO else 91  # se mide a los llegados en [CAL, DIAS)
# «La semana» son los días [PRIMER_DIA, FIN_DIA): el 19 al 25 de enero en modo normal.
PRIMER_DIA, FIN_DIA = (1, 8) if RAPIDO else (14, 21)
MIN_EN_RIESGO = 10  # el KM solo se fía donde quedan al menos tantos pacientes en riesgo

DIA = MINUTES_PER_DAY  # 1440 minutos
INICIO, FIN = CAL * DIA, DIAS * DIA  # la ventana de medida, en minutos
HORIZONTE = (DIAS + DRAIN_DAYS) * DIA  # con el día de vaciado de la calibración
NIVEL_COLOR = {1: "#d03b3b", 2: "#eb6834", 3: "#e3b505", 4: "#1baf7a", 5: "#2a78d6"}


def nombre_dia(d):
    """Nombre corto y fecha del día d de la ejecución (el día 0 es el lunes 5 de enero)."""
    fecha = LLEGADAS.start_date + timedelta(days=int(d))
    return f"{WEEKDAYS[fecha.weekday()][:3]} {fecha.day}"


def ejecuta(centro, perfiles, dias=DIAS, semilla=SEMILLA, cal=CAL):
    """Simula dias + el día de vaciado y devuelve la tabla de episodios, las curvas y el
    ground truth por episodio. Los eventos se liberan al salir de la función."""
    sink = MemorySink()
    resultado = simulate_ed(
        centro,
        LLEGADAS,
        dias + DRAIN_DAYS,
        semilla,
        sink=sink,
        warmup_days=cal,
        profiles=perfiles,
    )
    verdad = {t.episode_id: t for t in resultado.ground_truth.episodes}
    horizonte = (dias + DRAIN_DAYS) * DIA
    return episode_table(sink.events), step_curves(sink.events, float(horizonte)), verdad


def pct(x, decimales=1):
    """Un porcentaje con coma decimal, o un guion si no hay dato."""
    return "—" if x is None or np.isnan(x) else f"{100 * x:.{decimales}f} %".replace(".", ",")


plt.rcParams["figure.figsize"] = (9, 4)
print("Modo rápido" if RAPIDO else "Modo normal")

## 2. Las dos ejecuciones

Se simula el mismo hospital (llegadas de `config/llegadas.toml`, semilla 2026) dos veces: con el
modelo del hito 4, sin abandono, y con el del hito 5 calibrado. Los pacientes que llegan son los
mismos en las dos; lo que cambia es lo que les pasa. Las dos ejecuciones duran 91 días desde el
lunes 5 de enero de 2026, con 2 de calentamiento y un día de vaciado al final (el de la
calibración, para que los llegados al final de la ventana puedan salir).

Del hito 5 se guarda, además de la tabla de episodios y las curvas, el **ground truth**: lo que
se sorteó para cada paciente al llegar, incluida su paciencia y el destino que habría tenido
(pieza 7). Es lo que ningún hospital sabe, y lo que permite comprobar cada análisis.

In [ ]:
T4, C4, V4 = ejecuta(CENTRO_H4, PERFILES_H4)
T5, C5, V5 = ejecuta(CENTRO_H5, PERFILES_H5)

llegados = [e for e in T5 if INICIO <= e.arrival < FIN]
izq = left_without_being_seen_shares(T5, INICIO, FIN)
print(f"Hito 5: {len(llegados)} pacientes llegados en la ventana; se van sin ser vistos {izq.left}")
print(
    f"  ({pct(izq.share, 2)} de los llegados: {izq.before_triage} antes del triaje y"
    f" {izq.after_triage} después, según el sitio {dict(izq.by_wait)})"
)
print(f"Hito 4: {sum(INICIO <= e.arrival < FIN for e in T4)} llegados, nadie se va sin ser visto")

En la ejecución normal (91 días, semilla 2026) llegan 28.493 pacientes en la ventana y se van
sin ser vistos 1.061 (el 3,72 %): 26 antes del triaje, 555 esperando box y 480 esperando al
médico dentro del box. Con el hito 4 nadie se va, y es la cola de espera la que lo absorbe todo.

## 3. Abandono frente a saturación

**Pregunta:** ¿a qué hora se va más gente, y se va porque el servicio está saturado? Es el bucle del
diseño «saturación → abandono», que no se programó: sale solo de las colas, porque cada paciente
tiene una paciencia y la espera que le toca decide si la agota.

`abandonment_by_arrival_hour` agrupa a los llegados por la hora del día de su llegada (los 89 días
de la ventana se pliegan en uno). Para cada hora da el porcentaje que se va y dos medidas de la
saturación de esa hora:

- la **espera media de los atendidos** (reloj A: de la llegada al primer médico), que es la
  medida natural pero deja fuera a los que se fueron, que esperaron más;
- el **tiempo en cola de todos**: hasta ser visto o hasta irse, sin dejar fuera a nadie
  (salvo a quien sigue esperando al final).

In [ ]:
horas = abandonment_by_arrival_hour(T5, INICIO, FIN)
x = np.arange(24)
se_van = np.array([np.nan if h.left_share is None else 100 * h.left_share for h in horas])
espera = np.array([np.nan if h.mean_wait_attended is None else h.mean_wait_attended for h in horas])
cola = np.array([np.nan if h.mean_queue_time is None else h.mean_queue_time for h in horas])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(x, se_van, color=OBSERVED_COLOR, label="% que se va")
ax1.set_xlabel("hora de llegada")
ax1.set_ylabel("% de los llegados que se va sin ser visto")
ax1.set_xticks(range(0, 24, 3))
eje = ax1.twinx()
eje.plot(x, espera, color=THEORY_COLOR, marker="o", ms=4, label="espera media de los atendidos")
eje.plot(x, cola, color=SECOND_COLOR, marker="s", ms=4, label="tiempo en cola de todos")
eje.set_ylabel("minutos")
ax1.set_title("Abandono y espera por hora de llegada")
ax1.legend(loc="upper left", fontsize=8)
eje.legend(loc="upper center", fontsize=8)

ax2.scatter(cola, se_van, color=OBSERVED_COLOR)
for h in range(24):
    if not np.isnan(cola[h]):
        ax2.annotate(
            str(h), (cola[h], se_van[h]), fontsize=7, xytext=(3, 3), textcoords="offset points"
        )
ax2.set_xlabel("tiempo en cola de la hora (min)")
ax2.set_ylabel("% que se va")
ax2.set_title("Una hora por punto")
plt.tight_layout()
plt.show()

hay = ~np.isnan(se_van) & ~np.isnan(cola) & ~np.isnan(espera)
if hay.sum() >= 3:
    r_espera = np.corrcoef(espera[hay], se_van[hay])[0, 1]
    r_cola = np.corrcoef(cola[hay], se_van[hay])[0, 1]
    print(f"Correlación del % que se va con la espera de los atendidos: {r_espera:.2f}")
    print(f"Correlación del % que se va con el tiempo en cola de todos:  {r_cola:.2f}")
peor = int(np.nanargmax(se_van))
mejor = int(np.nanargmin(se_van))
print(
    f"Hora con más abandono: las {peor}h ({se_van[peor]:.1f} %, cola de {cola[peor]:.0f} min);"
    f" con menos: las {mejor}h ({se_van[mejor]:.1f} %, cola de {cola[mejor]:.0f} min)"
)

En la ejecución normal el abandono sigue a la cola casi punto por punto (correlación de 0,93 con la
espera de los atendidos y de 0,96 con el tiempo en cola de todos): de madrugada y a última hora
(de las 21h a la 1h) se va entre el 10 y el 15 % de los que llegan, con colas de 100 a 200 minutos;
a las 8 y las 9h, con colas de 11 a 15 minutos, se va el 0,1 al 0,3 %. La mañana trae más pacientes por hora que la noche, pero tiene 12 médicos por turno frente a 4 de
noche (la plantilla de la 4.6c, [12, 11, 4]), y sus colas son cortas.

Las dos medidas de saturación casi coinciden donde importa poco (por la mañana) y se separan
donde importa más (en las horas de mucho abandono, la de todos es mayor): **medir solo a los
atendidos subestima la saturación**, porque deja fuera a los que más esperaron y se fueron. Es el
sesgo del superviviente de la pieza 4.

**Para experimentar:** cambia `SEMILLA` y comprueba que el patrón por hora se mantiene aunque las
cifras exactas no; cambia `DIAS` a 28 y mira cómo se ensucia la curva de las horas con pocos
pacientes.

> **Lo que debes recordar:** el abandono es una realimentación negativa que se activa con la
> espera: las horas de mucha cola son las de más fugas, y lo que se ve de la espera (la de los
> atendidos) es siempre más corta que la que hubo.

## 4. Destinos frente a la gravedad real G

**Pregunta:** ¿el destino de cada paciente (alta, ingreso, traslado o exitus) depende de lo grave
que es de verdad?

El destino sale de una logística multinomial con el alta como referencia (decisión 71): sus
probabilidades dependen de la gravedad real G, no del nivel que le puso la enfermera ni de G + ε
(el ruido del triaje). Cuanto más grave, más ingreso. El paquete tiene **lo exacto**
(`OutcomeModel.by_severity`: la integral de la probabilidad sobre un año de pacientes, por
tramos de G con la misma masa) y el notebook lo compara con dos cosas simuladas:

- el destino **sorteado** de todos los pacientes, que está en el ground truth;
- el destino **observado** en el log, que solo existe para los que llegan a la decisión: quien se
  va no tiene destino.

Se compara siempre con lo exacto antes que con otra simulación.

In [ ]:
MODELO = outcome_model(PERFILES_H5, LLEGADAS)
bordes = MODELO.workup.severity_edges(5)  # de más grave a menos; G en (borde de abajo, de arriba]
exacto = MODELO.by_severity(5)
filas = destinations_by_severity(T5, V5, bordes, INICIO, FIN)

print(
    f"{'tramo de G':<18}{'llegados':>9}{'se van':>8}{'exacto':>9}{'sorteado':>10}{'observado':>11}"
)
print(f"{'':<35}% de ingreso")
for i, f in enumerate(filas):
    sorteado = f.drawn["ingreso"] / f.arrivals if f.arrivals else None
    atendidos = sum(f.observed.values())
    observado = f.observed["ingreso"] / atendidos if atendidos else None
    rango = f"({bordes[i + 1]:.2f}, {bordes[i]:.2f}]".replace("inf", "∞")
    exacto_i = exacto[i][DESTINATIONS.index("ingreso")]
    print(
        f"{rango:<18}{f.arrivals:>9}{f.left:>8}{pct(exacto_i):>9}{pct(sorteado):>10}{pct(observado):>11}"
    )

fig, ax = plt.subplots(figsize=(9, 4))
ancho = 0.27
posiciones = np.arange(len(filas))
for k, (nombre, color) in enumerate(
    (
        ("exacto (modelo)", THEORY_COLOR),
        ("sorteado (todos)", MUTED_COLOR),
        ("observado (log)", OBSERVED_COLOR),
    )
):
    valores = []
    for i, f in enumerate(filas):
        if k == 0:
            valores.append(100 * exacto[i][DESTINATIONS.index("ingreso")])
        elif k == 1:
            valores.append(100 * f.drawn["ingreso"] / f.arrivals if f.arrivals else np.nan)
        else:
            n = sum(f.observed.values())
            valores.append(100 * f.observed["ingreso"] / n if n else np.nan)
    ax.bar(posiciones + (k - 1) * ancho, valores, width=ancho, color=color, label=nombre)
ax.set_xticks(posiciones, ["más grave"] + [""] * (len(filas) - 2) + ["menos grave"])
ax.set_ylabel("% de ingreso")
ax.set_title("Ingreso por quintil de gravedad real G")
ax.legend()
plt.tight_layout()
plt.show()

# Un paciente concreto: las probabilidades exactas del modelo, sin simular nada.
razon = PERFILES_H5.reasons[0]
print(
    f"\nMotivo «{REASON_KEYS[0]}» (ingreso base {razon.admission_pct:g} %), joven, sin alteradas:"
)
for g in (-1.0, 0.0, 1.0, 2.0):
    p = destination_probabilities(PERFILES_H5.outcome, razon.admission_pct, 0, g, 0)
    print(
        f"  G = {g:+.1f}: "
        + "  ".join(f"{d} {100 * q:.1f} %" for d, q in zip(DESTINATIONS, p, strict=True))
    )

En la ejecución normal el ingreso baja de forma monótona con G: de un 25,5 % exacto (26,6 %
sorteado) en el quintil más grave a un 2,5 % (2,1 %) en el menos grave. Lo sorteado y lo exacto difieren en menos de dos puntos
(es el ruido de muestreo de unos 5.600 pacientes por tramo) y lo observado casi coincide con lo
sorteado: se van pocos (de 5 en el tramo más grave a 433 en el menos grave) y, como su destino
sorteado era casi siempre el alta, quitarlos apenas mueve el porcentaje. **Con un abandono
pequeño y de gente leve, lo observado es un buen espejo de lo exacto**; con más abandono de
gente grave, no lo sería.

La razón de usar G y no el nivel asignado: el nivel lo pone la enfermera con ruido (G + ε), y dos
pacientes con el mismo nivel asignado pueden tener G distintos. El destino de un infratriado es el
de su G alto, aunque lo hayan clasificado como un 4.

**Para experimentar:** pasa `severity_edges(5)` a `severity_edges(10)` y `by_severity(10)` para ver
diez tramos; el patrón se afila.

> **Lo que debes recordar:** el destino depende de la gravedad real G, no de lo que el hospital
> cree; lo exacto sirve de referencia para juzgar lo simulado, y lo observado solo es una parte de
> lo sorteado (los que se van no tienen destino).

## 5. Boxes bloqueados y ley de Little

**Pregunta:** ¿cuántos boxes ocupan los pacientes que ya están ingresados y esperan cama?

Un ingresado no libera su box hasta que le dan cama (H5-6): el box está *bloqueado* sin que un
médico lo use. La ley de Little dice que el número medio de pacientes en espera es la tasa de
peticiones por la espera media: **L = λ · W**. `bed_little` da los dos lados en una ejecución:

- **por Little:** λ son las peticiones de cama por minuto de los llegados de la ventana y W su
  espera media, con la de quien sigue esperando al final cortada en el horizonte;
- **por las curvas:** la media de `boxes_waiting_bed` (en su box) más la de `hallway` (los que un
  nivel 1 desalojó y esperan en el pasillo).

In [ ]:
little = bed_little(T5, C5, INICIO, FIN)
print(f"Peticiones de cama en la ventana: {little.requests} ({little.rate * DIA:.1f} al día)")
print(f"λ = {little.rate:.5f} peticiones por minuto")
mean_wait = little.mean_wait
print(f"W = {mean_wait:.1f} min de espera media ({mean_wait / 60:.1f} h)")
print(f"L por Little   = λ · W = {little.by_little:.3f} pacientes esperando cama")
print(
    f"L por las curvas = {little.boxes_mean:.3f} en su box + {little.hallway_mean:.4f} en el"
    f" pasillo = {little.by_curves:.3f}"
)
diferencia = abs(little.by_curves - little.by_little)
print(f"Diferencia: {diferencia:.3f} pacientes (los bordes de la ventana)")
bloqueados = pct(little.boxes_mean / CENTRO_H5.boxes)
print(f"De {CENTRO_H5.boxes} boxes, {bloqueados} están bloqueados de media")

a, b = PRIMER_DIA * DIA, FIN_DIA * DIA
pasos = curve_steps(C5, "boxes_waiting_bed", a, b)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.stairs(
    pasos.values, (pasos.edges - a) / DIA, color=OBSERVED_COLOR, label="boxes con un ingresado"
)
ax.axhline(little.by_little, color=THEORY_COLOR, ls="--", label=f"λ · W = {little.by_little:.2f}")
ax.set_xticks(
    range(FIN_DIA - PRIMER_DIA + 1),
    [nombre_dia(d) for d in range(PRIMER_DIA, FIN_DIA + 1)],
    fontsize=8,
)
ax.set_ylabel("boxes bloqueados")
ax.set_title("Boxes ocupados por pacientes que esperan cama, una semana")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

En la ejecución normal hay 2.940 peticiones de cama en los 89 días (unas 33 al día) con una espera
media de 326 min (5,4 h): **L = 0,02294 · 326,2 = 7,48 por Little frente a 7,49 por las curvas**
(7,494 en su box y 0,0008 en el pasillo). Cuadran: la diferencia, de una centésima, viene de que
los que piden cama antes de la ventana o la terminan después cuentan en una cuenta y no en la
otra. En la ventana anual de la 5.6 salió 6,50 por las curvas y 6,44 por λ · W. Unos 7,5 boxes
de 36 (el 21 %) están bloqueados de media: son los que se «pierden» para los pacientes nuevos,
y de ahí que la calibración derivara 36 boxes (30 + λ · W) para que el cuello de botella fueran
los médicos y no los boxes.

Que las dos formas de contar coincidan es también una **comprobación del motor**: si las
curvas contaran mal a los ingresados, Little no cerraría.

**Para experimentar:** mira cuándo salen los máximos del gráfico (la semana de enero tiene 16
boxes bloqueados a la vez como máximo). ¿Coinciden con los días de más abandono?

> **Lo que debes recordar:** `L = λ · W` conecta lo que se ve en una foto (cuántos esperan ahora)
> con lo que se ve en una película (cuánto espera cada uno); cuando no cuadra, hay un error de
> medida, no de azar.

## 6. La semana de enero, antes y después

**Pregunta:** ¿qué hace el hito 5 con la peor semana de enero de la ejecución principal del
notebook 04?

En la tarea 4.6c, la semana del 19 al 25 de enero de 2026 tenía esperas irreales: el reloj A del
nivel 5 llegaba a 3.897 minutos (unas 65 horas), porque sin abandono una cola que crece no se
corta nunca. Con el abandono, los que más esperan se van y la cola se acorta. Se mide con las
mismas ejecuciones de la sección 2 (91 días desde el 5 de enero; la semana son los días 14 a 20,
los llegados de esos días):

- `clock_mean`: la media del reloj A (de la llegada al primer médico) de los atendidos, para unos
  niveles;
- `left_without_being_seen_shares`: los que se van;
- `bed_occupancy`: los boxes bloqueados.

In [ ]:
a, b = PRIMER_DIA * DIA, FIN_DIA * DIA
inicio_semana, fin_semana = nombre_dia(PRIMER_DIA), nombre_dia(FIN_DIA - 1)
print(f"La semana: del {inicio_semana} al {fin_semana} (días {PRIMER_DIA} a {FIN_DIA - 1})\n")
print(f"{'':<34}{'Hito 4':>12}{'Hito 5':>12}")
semana = {}
for nombre, tabla, curvas in (("Hito 4", T4, C4), ("Hito 5", T5, C5)):
    semana[nombre] = {
        "A25": clock_mean(tabla, PRIMER_DIA, FIN_DIA, (2, 3, 4, 5)),
        "A5": clock_mean(tabla, PRIMER_DIA, FIN_DIA, (5,)),
        "izq": left_without_being_seen_shares(tabla, a, b),
        "camas": bed_occupancy(curvas, a, b),
    }


def celda(f, decimales=1):
    return f"{f:.{decimales}f}".replace(".", ",") if f is not None else "—"


s4, s5 = semana["Hito 4"], semana["Hito 5"]
print(f"{'Reloj A, niveles 2 a 5 (min)':<34}{celda(s4['A25'].mean):>12}{celda(s5['A25'].mean):>12}")
print(f"{'Reloj A, nivel 5 (min)':<34}{celda(s4['A5'].mean):>12}{celda(s5['A5'].mean):>12}")
print(f"{'  pacientes de ese nivel':<34}{s4['A5'].n:>12}{s5['A5'].n:>12}")
print(f"{'Abandonos':<34}{s4['izq'].left:>12}{s5['izq'].left:>12}")
print(f"{'  de los llegados':<34}{s4['izq'].arrivals:>12}{s5['izq'].arrivals:>12}")
print(f"{'  porcentaje':<34}{pct(s4['izq'].share, 2):>12}{pct(s5['izq'].share, 2):>12}")
camas4, camas5 = s4["camas"], s5["camas"]
print(f"{'Boxes con ingresados, media':<34}{celda(camas4.boxes_mean, 2):>12}", end="")
print(f"{celda(camas5.boxes_mean, 2):>12}")
print(f"{'  máximo':<34}{s4['camas'].boxes_max:>12}{s5['camas'].boxes_max:>12}")
if not RAPIDO:
    print(
        "\nRegistro de la 5.6 (docs/calibracion/registro.md): 306,0 y 3897,3 min con el hito 4;"
        " 101,0 y 434,7 min, 240 abandonos de 2549 (9,42 %), 7,65 boxes de media y 16 de máximo"
        " con el hito 5."
    )

dias = list(range(PRIMER_DIA, FIN_DIA))
fig, ax = plt.subplots(figsize=(10, 3.8))
for k, (nombre, tabla, color) in enumerate(
    (("Hito 4", T4, THEORY_COLOR), ("Hito 5", T5, OBSERVED_COLOR))
):
    valores = []
    for d in dias:
        m = clock_mean(tabla, d, d + 1, (2, 3, 4, 5)).mean
        valores.append(np.nan if m is None else m)
    ax.bar(np.arange(len(dias)) + (k - 0.5) * 0.4, valores, width=0.4, color=color, label=nombre)
ax.set_xticks(range(len(dias)), [nombre_dia(d) for d in dias])
ax.set_ylabel("reloj A medio, niveles 2 a 5 (min)")
ax.set_title("La misma semana con y sin abandono")
ax.legend()
plt.tight_layout()
plt.show()

En la ejecución normal (días 14 a 20, semilla 2026) el reloj A medio de los niveles 2 a 5 baja de
**306,0 a 101,0 minutos** y el del nivel 5, de **3.897,3 a 434,7**. Se van 240 de los 2.549
pacientes de la semana (el 9,42 %), sobre todo de los niveles 4 y 5 en los días más saturados, y
hay 7,65 boxes con ingresados de media, 16 como máximo. Las cifras son exactamente las del
registro de la 5.6.

Dos cosas que conviene entender:

- **El abandono acota la cola, no la elimina.** Los 434,7 min del nivel 5 siguen siendo mucho:
  la plantilla es anual y el refuerzo de invierno queda para versiones posteriores.
- **La media de los atendidos sigue ignorando a los que se fueron.** Un día malo parecería mejor si
  solo se mirara esa media; por eso el cumplimiento de Manchester cuenta como incumplimiento al
  que se va después de su máximo (pieza 4).

**Para experimentar:** cambia `PRIMER_DIA` y `FIN_DIA` a otra semana (por ejemplo, los días 56 a 63,
en primavera) y mira cuánto tiene que ver el abandono con la estación.

> **Lo que debes recordar:** sin abandono, una cola saturada crece sin límite; con él, la propia
> espera expulsa a los menos pacientes, y la media de los atendidos y el recuento de los que se
> van hay que leerlos juntos.

## 7. El contrafactual de quien se fue

**Pregunta:** ¿cuántos de los que se fueron habrían ingresado? Ningún hospital lo sabe, porque
quien se va no vuelve a pasar por la decisión (pieza 7). EDsim sí: el ground truth guarda el
destino que se sorteó para todos los pacientes al llegar, también para los que se fueron, que es
el que habrían tenido. Es un «resultado potencial» de la inferencia causal que en datos reales
nunca se observa.

`abandonment_counterfactual` compara, por nivel asignado, el ingreso de los **que se quedaron y
salieron** (observado en el log) con el de los **que se fueron** (el destino sorteado). La fila
«antes del triaje» agrupa a quien se fue sin que nadie le asignara nivel.

In [ ]:
filas_cf = abandonment_counterfactual(T5, V5, INICIO, FIN)
etiquetas = [str(f.level) if f.level else "antes\ndel triaje" for f in filas_cf]
print(f"{'nivel':<14}{'se quedaron':>12}{'ingresan':>10}{'se fueron':>11}{'habrían ingresado':>19}")
for f in filas_cf:
    nivel = str(f.level) if f.level else "antes triaje"
    print(
        f"{nivel:<14}{f.stayed:>12}{pct(f.stayed_share):>10}{f.left:>11}"
        f"{f.left_would_admit:>6} ({pct(f.left_share)})"
    )

fig, ax = plt.subplots(figsize=(9, 4))
pos = np.arange(len(filas_cf))
for k, (nombre, color, valores) in enumerate(
    (
        ("se quedaron (observado)", OBSERVED_COLOR, [f.stayed_share for f in filas_cf]),
        ("se fueron (destino sorteado)", THEORY_COLOR, [f.left_share for f in filas_cf]),
    )
):
    ax.bar(
        pos + (k - 0.5) * 0.4,
        [np.nan if v is None else 100 * v for v in valores],
        width=0.4,
        color=color,
        label=nombre,
    )
ax.set_xticks(pos, etiquetas)
ax.set_xlabel("nivel asignado")
ax.set_ylabel("% que ingresa (o habría ingresado)")
ax.set_title("El ingreso de los que se quedaron frente al de los que se fueron")
ax.legend()
plt.tight_layout()
plt.show()

verdad_abandono = true_level_abandonment(T5, V5, INICIO, FIN)
habrian = sum(f.left_would_admit for f in filas_cf)
se_fueron = sum(f.left for f in filas_cf)
parte = pct(habrian / se_fueron if se_fueron else None)
print(f"\nSe fueron {se_fueron} y habrían ingresado {habrian} ({parte})")
print(
    f"Comprobación con el ground truth por nivel real: {verdad_abandono.left} se fueron,"
    f" {verdad_abandono.would_be_admitted} habrían ingresado,"
    f" {verdad_abandono.severe} era grave de verdad"
)
assert habrian == verdad_abandono.would_be_admitted and se_fueron == verdad_abandono.left

En la ejecución normal se van 1.061 pacientes de la ventana y **55 habrían ingresado (el 5,2 %)**;
solo 1 era grave de verdad (nivel real 1 o 2). Por nivel asignado, el ingreso de los que se
quedaron es del 61 % en el nivel 1, el 30 % en el 2, el 12,5 % en el 3, el 4,2 % en el 4 y el 2,1 %
en el 5; el de los que se fueron (con su destino sorteado) es del mismo orden: 10,3 %, 5,2 % y
2,3 % en los niveles 3, 4 y 5. Se van los que tienen poca paciencia, no los que tienen menos
riesgo, así que **abandonar no elimina el riesgo de ingreso, solo lo esconde**: 55 pacientes
de esa ventana habrían necesitado una cama y nadie los vio. En el año completo de la 5.6 fueron
23 de los 729 que se fueron.

La suma coincide con la que da el ground truth por nivel real (`would_be_admitted`), calculada
por otro camino: la celda lo comprueba con un `assert`.

**Para experimentar:** cambia `SEMILLA` y mira cuánto varía entre semillas el número de los que
habrían ingresado (55 con la 2026); es una cuenta de pocos casos y se mueve.

> **Lo que debes recordar:** comparar el ingreso de los que se quedaron con el de los que se
> fueron sin conocer el destino de los segundos sería un error clásico de inferencia; EDsim lo
> tiene todo, porque sortea el destino al llegar.

## 8. Kaplan–Meier de la paciencia

**Pregunta:** si solo vemos el log, ¿cuánto aguanta la gente? Lo que se observa de los que se
van **no es su paciencia**: solo se van los de poca paciencia, y a los demás los ve un médico
antes de agotarla. El estimador de Kaplan–Meier (KM) es la herramienta clásica para esto: el
abandono es el *evento*, y ver al médico o seguir esperando al final es una *censura* (del
paciente solo se sabe que su paciencia era **mayor** que lo que esperó).

Primero un ejemplo a mano con `kaplan_meier`. Tiempos 10 (evento), 20 (censura), 30 (evento) y 40
(evento):

In [ ]:
km = kaplan_meier([10, 20, 30, 40], [True, False, True, True])
for t, riesgo, supervivencia in zip(km.times, km.at_risk, km.survival, strict=True):
    print(f"t = {t:>4.0f}: en riesgo {riesgo}, S = {supervivencia:.3f}")
print("Con un empate evento-censura en el mismo tiempo, el evento va antes:")
empate = kaplan_meier([10, 20, 20, 30], [True, True, False, True])
print("  S =", [round(s, 3) for s in empate.survival], " en riesgo", empate.at_risk)

S(10) = 3/4 = 0,75. El censurado en 20 sale del riesgo sin haberse ido: en 30 quedan 2 y
S(30) = 0,75 · 1/2 = 0,375; en 40 queda uno y S(40) = 0. Con el empate de un evento y una
censura en el mismo tiempo, el censurado sigue en riesgo en ese instante (el evento va antes, el
convenio habitual): S(20) = 0,75 · 2/3 = 0,5.

Ahora con la ejecución de verdad. `patience_survival` hace el KM por nivel, con el tiempo de cada
paciente desde su llegada: evento si se va, censura en el instante en que lo ve un médico (o en
el horizonte si sigue esperando). Se compara con la **supervivencia empírica de las paciencias
verdaderas** del ground truth de ese mismo grupo (no con una sola lognormal: con la edad la
paciencia de un nivel es una mezcla de lognormales). Se hace por nivel real (del que depende
la paciencia) y, aparte, por nivel asignado (lo que vería el hospital).

In [ ]:
SUP = patience_survival(T5, V5, INICIO, FIN, HORIZONTE, "true_level")
SUP_ASIGNADO = patience_survival(T5, V5, INICIO, FIN, HORIZONTE, "assigned_level")
rejilla = np.linspace(0, 1500, 301)


def dibuja(ax, sup, titulo):
    """KM sólido donde quedan al menos MIN_EN_RIESGO pacientes, gris donde no, y la verdad."""
    for g in sup.groups:
        if g.n == 0 or g.left == 0:
            continue
        color = NIVEL_COLOR[g.level]
        km_ = np.array([g.km.at(t) for t in rejilla])
        riesgo = np.array([g.km.at_risk_at(t) for t in rejilla])
        fiable = riesgo >= MIN_EN_RIESGO
        ax.plot(rejilla[fiable], km_[fiable], color=color, label=f"nivel {g.level} (KM)")
        ax.plot(rejilla[~fiable], km_[~fiable], color=color, alpha=0.25)
        verdad = np.array([g.truth_survival(t) for t in rejilla])
        ax.plot(rejilla, verdad, color=color, ls="--", lw=1, label=f"nivel {g.level} (verdad)")
    ax.set_xlabel("minutos desde la llegada")
    ax.set_ylabel("S(t) = P(paciencia > t)")
    ax.set_ylim(0, 1.02)
    ax.set_title(titulo)
    ax.legend(fontsize=7, ncol=2)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
dibuja(ax1, SUP, "Por nivel real: el KM sigue a la verdad")
dibuja(ax2, SUP_ASIGNADO, "Por nivel asignado (lo que ve el hospital)")
plt.tight_layout()
plt.show()

print(
    f"{'nivel real':<11}{'n':>7}{'se van':>8}{'ven médico':>12}{'esperan':>9}{'plazo vencido':>15}"
)
for g in SUP.groups:
    print(f"{g.level:<11}{g.n:>7}{g.left:>8}{g.seen:>12}{g.waiting:>9}{g.late_leavers:>17}")
print(
    f"\nPor nivel asignado quedan fuera {SUP_ASIGNADO.unclassified} que se fueron antes del triaje."
)

In [ ]:
TIEMPOS = (60, 120, 240, 480, 960)
print(f"KM ± error típico (Greenwood) / verdad, donde quedan al menos {MIN_EN_RIESGO} en riesgo:")
print(f"{'':<8}" + "".join(f"{'t = ' + str(t):>22}" for t in TIEMPOS))
desvios = []
for g in SUP.groups:
    if g.left < 30:
        continue
    celdas = []
    for t in TIEMPOS:
        if g.km.at_risk_at(t) >= MIN_EN_RIESGO:
            km_t, error, verdad_t = g.km.at(t), g.km.standard_error_at(t), g.truth_survival(t)
            celdas.append(f"{km_t:.3f}±{error:.3f} / {verdad_t:.3f}")
            if error > 0:
                desvios.append(((km_t - verdad_t) / error, g.level, t))
        else:
            celdas.append("pocos en riesgo")
    print(f"nivel {g.level:<2}" + "".join(f"{c:>22}" for c in celdas))
if desvios:
    z, nivel, t = max(desvios, key=lambda d: abs(d[0]))
    fuera = sum(abs(d[0]) > 2 for d in desvios)
    print(
        f"\nMayor desvío: nivel {nivel}, t = {t}: {z:+.1f} errores típicos."
        f" Celdas a más de 2 errores típicos: {fuera} de {len(desvios)}"
    )

In [ ]:
def mediana_km(g):
    return next((t for t, s in zip(g.km.times, g.km.survival, strict=True) if s <= 0.5), None)


print("Mediana de la paciencia, nivel real 4:")
g4 = SUP.groups[3]
vistos = [
    e.time_to_leave
    for e in T5
    if INICIO <= e.arrival < FIN
    and e.left is not None
    and V5[e.episode_id].clinical.triage.true_level == 4
]
if vistos and g4.n:
    km_mediana = mediana_km(g4) or float("nan")
    print(f"  mediana de espera de los que se fueron (lo que se ve): {np.median(vistos):.0f} min")
    print(f"  Kaplan–Meier:                                  {km_mediana:.0f} min")
    print(f"  paciencia verdadera de todos (ground truth):   {np.median(g4.patiences):.0f} min")
print(
    "  mediana del perfil sin ajuste de edad:",
    f"{patience_quantile(PERFILES_H5.outcome.abandonment, 4, 1, 0.5):.0f} min",
)
tardios = sum(g.late_leavers for g in SUP.groups)
se_fueron = sum(g.left for g in SUP.groups)
print(
    f"\nSe van con el plazo vencido durante el triaje (tiempo mayor que su paciencia): {tardios}"
    f" de {se_fueron} ({pct(tardios / se_fueron if se_fueron else None)})"
)

En la ejecución normal el KM por nivel real **sigue a la supervivencia verdadera**. En el nivel 4
(14.157 pacientes, 927 que se van) da S(120) = 0,956 ± 0,003 frente a 0,955 de la verdad,
S(240) = 0,830 ± 0,007 frente a 0,822 y S(480) = 0,525 ± 0,015 frente a 0,549 (364 en riesgo). De
las 13 celdas de la tabla con al menos 10 pacientes en riesgo, una está a más de 2 errores
típicos (el nivel 3 a los 480 min, con −2,5, y 61 en riesgo); con 13 comparaciones, una así no
sorprende. Que el KM acierte con solo el log es porque la censura (ver al médico) no depende de
la paciencia de cada paciente: quien espera más no es quien tiene más paciencia.

Lo que se ve de los que se van es otra cosa: su espera mediana es de **228 minutos**, el KM estima
una mediana de **504** y la paciencia verdadera de los 14.157 pacientes del nivel es de **534**.
Quien se va lo hace con poca paciencia; **la paciencia de todos es mucho mayor** (pieza 1).

Tres lecciones del gráfico:

- **El KM solo estima donde quedan pacientes en riesgo.** Por encima del mayor tiempo observado no
  estima nada y la curva queda plana; con pocos en riesgo es muy ruidosa. Por eso el gráfico
  dibuja en gris los tramos con menos de 10 en riesgo y la tabla no compara ahí. En el nivel 2,
  por ejemplo, a los 240 min quedan solo 13 en riesgo y a los 480, ninguno: su supervivencia más
  allá no se ve.
- **Con esperas cortas, el abandono informa de la cola izquierda de la paciencia, no de su
  mediana.** Con k = 4 la mediana de la paciencia del nivel 4 es de unos 600 min y casi nadie
  espera tanto (el mayor tiempo observado del nivel 4 es de 1.611 min, con 14 en riesgo a los
  960): lo que el abandono permite medir es la parte baja de la distribución, y por eso la
  calibración de la 5.6 necesitó k = 4 para llegar al abandono observado.
- **Por nivel asignado el KM sigue a la verdad de su propio grupo** (el panel de la derecha), pero
  ese grupo no es un nivel real: mezcla pacientes de varios niveles reales, y los 26 que se
  fueron antes del triaje no tienen nivel y quedan fuera. La paciencia «del nivel 4 asignado» no
  es la del nivel real 4: lo que el hospital estima con su nivel describe a otra gente que la que
  EDsim puede estudiar con el nivel real.

El KM trata como evento, en su tiempo observado, a quien tenía el plazo vencido durante el triaje y
se va al acabarlo (H5-4): su tiempo observado es mayor que su paciencia. Son pocos, **7 de
1.061 (el 0,7 %)**, y el estimador no los corrige: simplemente los traslada algo a la derecha.

**Para experimentar:** sube `MIN_EN_RIESGO` a 50 y mira cuánto se acorta el tramo que se fía del
KM; cambia `"true_level"` por `"assigned_level"` en la tabla y compara.

> **Lo que debes recordar:** lo observado de los que se van no es su paciencia; el KM la estima
> tratando como censura a quien ve al médico, pero solo donde queda gente en riesgo. Lo que el
> abandono enseña es la cola izquierda de la paciencia.

## 9. Resumen

Lo que debes recordar de cada sección:

3. El abandono sigue a la espera hora a hora (correlación de 0,93 a 0,96); medir la espera solo
   de los atendidos subestima la saturación.
4. El destino depende de G, no del nivel asignado; lo exacto es la referencia, lo sorteado lo
   acompaña y lo observado es solo una parte (los que se van no tienen destino).
5. `L = λ · W`: 7,48 por Little frente a 7,49 por las curvas, con unos 7,5 de los 36 boxes
   bloqueados por ingresados.
6. En la semana de enero el reloj A del nivel 5 baja de 3.897,3 a 434,7 min con el abandono; el
   abandono acota la cola, no la elimina.
7. De los 1.061 que se fueron, 55 habrían ingresado (el 5,2 %): abandonar esconde el riesgo.
8. El KM sigue a la supervivencia verdadera donde quedan pacientes en riesgo; lo observado de los
   que se van (228 min) no es su paciencia (534 min de mediana).

Siguiente: el notebook [06](06_tiempo_real.ipynb) lleva este mismo modelo a tiempo real con
Kafka.

In [ ]:
pico = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024  # en Linux, KB → MB
total = time.perf_counter() - INICIO_NB
print(f"Tiempo total: {total:.0f} s; pico de memoria del kernel: {pico:.0f} MB")